In [85]:

import json
import os

In [86]:
train_datasets = [
  # 텍스트형 1
  "마쓰리의_나라_신의_나라(Page15)",
  "어느덧_40년입니다(Page3)",
  "슬기로운_빵지순례(Page8)",
  "KBB(일본_캐드_전자책)(Page14)",
  "진주를_담은_전자책_만들기_진주남강유등축제(Page23)",
  "surabaya_food(Page4)",
  
  # 이미지형 7
  "그림모음집(Page4)",
  "i3_아이빌리브(Page4)",
  "진주를_담은_전자책_만들기_진주남강유등축제(Page22)",
  "최세경작품집2(Page1)",
  "최세경작품집2(Page7)",
  "최세경작품집2(Page9)", 
  
  # 슬라이더형 13
  "Internet_Advertising(Page10)",
  "진주를_담은_전자책_만들기_진주남강유등축제(Page34)",
  "대구교육_Vol.79_(ePUB3.0)(Page9)",
  "최세경작품집2(Page4)",
  "Blackpink_In_Your_Area(Page6)",
  "Exploring_Bogor_Indonesia(Page4)",
  "Indonesian_Food(Page5)",
  
  # 아코디언형 20
  "서초구청-업무매뉴얼(Page37)",
  "숨겨진_보석을_찾아서_울릉도,_독도(Page15)",
  "Pesona_Cirebon(Page6)",
  "Pesona_Cirebon(Page7)",
  "Wonderful_Indonesia_-_Wonderful_Journey(Page7)",
  "Skincare_for_Beginners(Page3)",
  "Tempat_Wisata_di_Buton_Selatan(Page3)",
  
  # 탭형 27
  "tab_4(Page1)",
  "열린_한국어_초급(Page6)",
  "4_Top_Movies 2023(Page5)",
  "Ekonomi_Makro(Page19)",
  "엘리스(Page4)",
  "최세경작품집2(Page26)",
  "Blackpink_In_Your_Area(Page4)",
  
  # 컴플렉스형 34
  "complex_4(Page3)",
  "complex_5(Page3)",
  "Pesona_Cirebon(Page4)",
  "대구교육_Vol.79_(ePUB3.0)(Page42)",
  "진주를_담은_전자책_만들기_진주남강유등축제(Page8)",
  "Musiqi_4(Page10)",
  "i3_아이빌리브(Page9)"
]

In [87]:
validation_datasets = [
  # 텍스트형
  "시니어모델워킹매력(Page3)",
  "Yogyakarta_Beaches(Page3)",
  "text_6(Page2)",
  
  # 이미지형
  "슬기로운_빵지순례(Page8)",
  "최세경작품집2(Page8)",
  "Musiqi_4(Page7)",
  
  # 슬라이더형
  "벽화는_사랑을_싣고(Page14)",
  "숨겨진_보석을_찾아서_울릉도,독도(Page8)",
  "Yogyakarta_Beaches(Page4)",
  "진주를_담은_전자책_만들기_진주남강유등축제(Page9)",
  "KBB(일본_캐드_전자책)(Page8)",
  "bahasa_indonesia_수정(Page7)",
  
  # 아코디언형
  "필기통(Page5)",
  "bahasa_indonesia_수정(Page3)",
  "비피랩_카탈로그(Page4)",
  "숨겨진_보석을_찾아서_울릉도,독도(Page4)",
  "서초구청_업무메뉴얼(Page1)",
  "Pesona_Cirebon(Page5)",
  
  # 탭형
  "tab_4(Page2)",
  "tab_4(Page3)",
  "Streetsea.id(Page4)",
  "언어를_공부합시다(Page5)",
  "엘리스(Page4)",
  "진주를_담은_전자책_만들기_진주남강유등축제(Page27)",
  
  # 컴플렉스형
  "complex_4(Page1)",
  "complex_4(Page2)",
  "complex_5(Page1)",
  "complex_5(Page2)",
  "비피랩_카탈로그(Page1)",
  "비피랩_카탈로그(Page7)"
]

In [88]:
system_prompt = """
다음의 지시사항에 따라 주어진 manuscript를 symbolic tree 구조의 JSON 형태로 변환하세요.

1. **모델의 출력 형태/목적**
    - 너는 주어진 manuscript를 입력받아, 이를 symbolic tree 구조(**JSON 형식**)로 변환하는 역할을 수행한다.
    - 그 외 이미지나 텍스트에 대한 불필요한 묘사나 설명은 일절 하지 않는다.
    - 오로지 symbolic tree로 변환하기 위해 필요한 최소한의 식별 정보만을 활용한다. (예: 부모 노드일 경우symbol type, direction 그리고 리프 노드일 경우 symbol type, content id)
    
2. **Task Description**
    - 전자책 레이아웃이란 한 페이지 안에서 콘텐츠(이미지 및 텍스트)들의 배치 관계를 의미한다.
    - manuscript는 저자가 보여주고 싶은 순서로 콘텐츠를 나열한 목록이다.
    - 페이지의 콘텐츠는 container, widget 으로 중첩할 수 있으며, 이 중첩 구조를 symbolic tree로 표현한다.
    - symbolic tree에서 부모 노드는 자식 노드를 **어떤 방향으로 배치할지를 결정**하며 left2right, top2down 값을 가질 수 있다.
    - 자식 노드는 container나 widget(layerlist_tab, layerlist_arccodian, layerlist_slider) 혹은 콘텐츠 자체일 수 있다.
    - 리프 노드는 실제 콘텐츠(텍스트, 이미지, 아이콘 등)를 직접 나타내는 단일 객체이다.
    
3. **Symbolic Tree 노드 속성**
    - **부모 노드(컨테이너/위젯) 속성**
        
        
        | symbol type | direction | description |
        | --- | --- | --- |
        | container | top2down / left2right | 다른 container나 위젯, 그리고 리프 콘텐츠를 포함 가능 |
        | textbox | left2right | text 타입 콘텐츠만 포함 가능 |
        | layerlist_tab | top2down | 탭을 눌러 자식 콘텐츠(화면에 보여지는 콘텐츠)를 교체할 수 있음 |
        | layerlist_arccodion | top2down | 탭을 눌러 자식 콘텐츠를 교체할 수 있음 |
        | layerlist_slider | left2right | 탭을 눌러 자식 콘텐츠를 교체할 수 있음 |
        | nac_title | left2right | layerlist 위젯의 자식 제목 텍스트를 담는 역할 |
        | nac_item | top2down / left2right | layerlist 위젯의 자식 항목 콘텐츠를 담으며, 컨테이너처럼 동작 |
    - **리프 노드(콘텐츠) 속성**
        
        
        | symbol type | direction | description |
        | --- | --- | --- |
        | image | - | 일반 크기의 삽화, 사진 등 |
        | icon | - | 작은 크기의 그래픽, 사용자가 누르는 버튼 또는 선택 요소 |
        | text | - | 일반적인 문장이나 설명 텍스트 |
        | title | - | 섹션을 대표하는 짧은 문구, 일반적으로 섹션의 가장 상단에 표시 |
        
4. **Manuscript 형식**
    - manuscript에서는 콘텐츠의 아이디(리프 노드의 symbol type + index), 그리고 그 아이디에 대응되는 실제 내용이 개행되어 한 쌍씩 주어진다.
    - manuscript의 예시는 다음과 같다.
        
        ```
        {"type": "text", "text": "text001"}, {"type": "text", "text": "BASEA.KU BY STREETSEA.ID"}, {"type": "text", "text": "image001"}, {"type": "image_url", "image_url": {"url": "https://d1bzdv1wm9phyk.cloudfront.net/local/epub/19594/OEBPS/nep_image/home_button-removebg-preview.png"}}
        ```
        
    
5. Symbolic Tree 형식
    - symbolic tree 구성 시 **부모 노드일 때 container 또는 widget 으로 자식 노드의 배치 방향을 direction으로 결정**한다.
    - symbolic tree 구성 시 **리프 노드일 때 콘텐츠의 아이디를 참조하며 manuscript의 콘텐츠 내용을 직접 출력해서는 안된다**.
    - symbolic tree의 예시는 다음과 같다.
    
    ```
    {\"symbol_name\": \"container\", \"direction\": \"top2down\", \"children\": [{\"symbol_name\": \"container\", \"direction\": \"left2right\", \"children\": [{\"symbol_name\": \"textbox\", \"direction\": \"top2down\", \"children\": [\"text001\"]}, \"image001\"]}
    ```
    
6. **출력 시 주의사항**
    - 본 시스템 지시문에 언급되지 않은 임의의 설정·지침·내용을 생성하거나 설명하지 않는다.
    - 요청사항이 주어진 경우에도, 시스템 지시문과 충돌하는 내용(예: 이미지를 시각적으로 묘사해 달라)은 **반드시 거부**하거나 제한된 범위 내에서만 수행한다.
"""

In [89]:
all_conversations_messages = []

for folder_name in train_datasets:
    manuscript_path = os.path.join("datasets", folder_name, "title_ver", "manuscript.json")
    symbolic_tree_path = os.path.join("datasets", folder_name, "title_ver", "symbolic_tree.json")

    # 파일 불러오기
    with open(manuscript_path, "r", encoding="utf-8") as file:
        manuscript = json.load(file)

    with open(symbolic_tree_path, "r", encoding="utf-8") as file:
        symbolic_tree = json.load(file)

    # system 메시지 생성 (빈 값)
    system_message = {
        "role": "system",
        "content": system_prompt
    }

    # user 메시지에 manuscript 데이터 추가
    user_message = {
        "role": "user",
        "content": []
    }
    
    for symbol_name, content in manuscript.items():
        if content["type"] in ["icon", "image"]:
            type = "image_url"
            content = {
                "url": content["content"],
                "detail": "low",
            }
        else:
            type = "text"
            content = content["content"]
        
        user_message["content"].append({"type": "text", "text": symbol_name})
        user_message["content"].append({"type": type, type: content})
    
    # assistant 메시지에 symbolic_tree 데이터 추가
    assistant_message = {
        "role": "assistant",
        "content": json.dumps(symbolic_tree, ensure_ascii=False)
    }

    # 메시지 리스트 생성
    messages = {"messages": [system_message, user_message, assistant_message]}
    all_conversations_messages.append(messages)
all_conversations_messages

[{'messages': [{'role': 'system',
    'content': '\n다음의 지시사항에 따라 주어진 manuscript를 symbolic tree 구조의 JSON 형태로 변환하세요.\n\n1. **모델의 출력 형태/목적**\n    - 너는 주어진 manuscript를 입력받아, 이를 symbolic tree 구조(**JSON 형식**)로 변환하는 역할을 수행한다.\n    - 그 외 이미지나 텍스트에 대한 불필요한 묘사나 설명은 일절 하지 않는다.\n    - 오로지 symbolic tree로 변환하기 위해 필요한 최소한의 식별 정보만을 활용한다. (예: 부모 노드일 경우symbol type, direction 그리고 리프 노드일 경우 symbol type, content id)\n    \n2. **Task Description**\n    - 전자책 레이아웃이란 한 페이지 안에서 콘텐츠(이미지 및 텍스트)들의 배치 관계를 의미한다.\n    - manuscript는 저자가 보여주고 싶은 순서로 콘텐츠를 나열한 목록이다.\n    - 페이지의 콘텐츠는 container, widget 으로 중첩할 수 있으며, 이 중첩 구조를 symbolic tree로 표현한다.\n    - symbolic tree에서 부모 노드는 자식 노드를 **어떤 방향으로 배치할지를 결정**하며 left2right, top2down 값을 가질 수 있다.\n    - 자식 노드는 container나 widget(layerlist_tab, layerlist_arccodian, layerlist_slider) 혹은 콘텐츠 자체일 수 있다.\n    - 리프 노드는 실제 콘텐츠(텍스트, 이미지, 아이콘 등)를 직접 나타내는 단일 객체이다.\n    \n3. **Symbolic Tree 노드 속성**\n    - **부모 노드(컨테이너/위젯) 속성**\n        \n        \n        | symbol type | direction | description

In [90]:
# JSONL 파일로 저장
file_path = 'train-40.jsonl'

with open(file_path, 'w', encoding='utf-8') as f:
    for conversation in all_conversations_messages:
        json_line = json.dumps(conversation, ensure_ascii=False)
        f.write(json_line + '\n')